# Daily Appointment Data Generator

## Goal

Use `vcu_master_daily.csv` to create synthetic appointment/event records for every admission counted on every day in the dataset. Each generated record has a unique ID, its source date and admission category, and a synthetic check-in time linked back to the daily source row.

## Arrival-time assumption

`vcu_master_daily.csv` contains daily aggregate admission counts, not observed appointment-level arrivals or hourly patterns. Check-in times are sampled from a seeded 24-hour sinusoidal intensity curve that peaks at 10:00 and declines toward a 22:00 trough. The same broad curve is applied to all categories, and daily category counts are preserved. This creates a daytime cluster for prototype runs; it is not a measured VCU arrival pattern.

## Patient attributes (Phase 1)

Each record also gets a synthetic `severity`, a queue `priority_rank`, `is_moveable`, `move_window_days`, `deadline_date`, and a staffing `unit`. All are assumptions from `patient_attributes` in `data/assumptions.yaml`; no patient-level severity data exists.

In [2]:
import os
import sys
from pathlib import Path

# Archived under notebooks/archive/; run from the repo root so data/ and src/ resolve.
repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(repo_root)
sys.path.insert(0, str(repo_root))

In [3]:
from pathlib import Path
import pandas as pd

csv_path = Path("data") / "vcu_master_daily.csv"
df = pd.read_csv(csv_path)

df.head()

,date,total_staffed_beds,total_occupied_beds,total_free_beds,occupied_beds_newborn,occupied_beds_delivery,occupied_beds_non_elective_via_ed,occupied_beds_non_elective_other,occupied_beds_elective,occupied_beds_covid_19,...,expected_discharges_other_viral,expected_discharges_intensive,avg_los_newborn_days,avg_los_delivery_days,avg_los_non_elective_via_ed_days,avg_los_non_elective_other_days,avg_los_elective_days,avg_los_covid_19_days,avg_los_influenza_days,avg_los_other_viral_days
0,2017-01-01,905,660.0,245.0,43,47,100,120,76,NaN,...,18.898079,16.090096,2.096227,2.604650,5.144324,6.663770,4.170857,NaN,4.591475,5.386618
1,2017-01-02,905,659.0,246.0,38,45,104,114,83,NaN,...,18.981292,15.397648,2.081085,2.523753,5.118284,6.379377,4.126970,NaN,4.651885,5.368373
2,2017-01-03,905,653.0,252.0,33,41,104,117,84,NaN,...,18.971664,15.838223,2.120760,2.518487,5.044225,6.367644,4.159663,NaN,4.680748,5.582925
3,2017-01-04,905,642.0,263.0,29,38,108,114,73,NaN,...,19.084184,15.696313,2.086112,2.588535,4.968106,6.465672,4.217788,NaN,4.643710,5.499093
4,2017-01-05,905,652.0,253.0,29,34,103,121,75,NaN,...,17.747718,16.763519,2.124785,2.665197,4.989300,6.521617,4.230880,NaN,4.624938,5.570379


In [4]:
df["key"] = range(1, len(df) + 1)
df.head()

,date,total_staffed_beds,total_occupied_beds,total_free_beds,occupied_beds_newborn,occupied_beds_delivery,occupied_beds_non_elective_via_ed,occupied_beds_non_elective_other,occupied_beds_elective,occupied_beds_covid_19,...,expected_discharges_intensive,avg_los_newborn_days,avg_los_delivery_days,avg_los_non_elective_via_ed_days,avg_los_non_elective_other_days,avg_los_elective_days,avg_los_covid_19_days,avg_los_influenza_days,avg_los_other_viral_days,key
0,2017-01-01,905,660.0,245.0,43,47,100,120,76,NaN,...,16.090096,2.096227,2.604650,5.144324,6.663770,4.170857,NaN,4.591475,5.386618,1
1,2017-01-02,905,659.0,246.0,38,45,104,114,83,NaN,...,15.397648,2.081085,2.523753,5.118284,6.379377,4.126970,NaN,4.651885,5.368373,2
2,2017-01-03,905,653.0,252.0,33,41,104,117,84,NaN,...,15.838223,2.120760,2.518487,5.044225,6.367644,4.159663,NaN,4.680748,5.582925,3
3,2017-01-04,905,642.0,263.0,29,38,108,114,73,NaN,...,15.696313,2.086112,2.588535,4.968106,6.465672,4.217788,NaN,4.643710,5.499093,4
4,2017-01-05,905,652.0,253.0,29,34,103,121,75,NaN,...,16.763519,2.124785,2.665197,4.989300,6.521617,4.230880,NaN,4.624938,5.570379,5


In [4]:
df.loc[df["key"] == 1]


,date,total_staffed_beds,total_occupied_beds,total_free_beds,occupied_beds_newborn,occupied_beds_delivery,occupied_beds_non_elective_via_ed,occupied_beds_non_elective_other,occupied_beds_elective,occupied_beds_covid_19,...,expected_discharges_intensive,avg_los_newborn_days,avg_los_delivery_days,avg_los_non_elective_via_ed_days,avg_los_non_elective_other_days,avg_los_elective_days,avg_los_covid_19_days,avg_los_influenza_days,avg_los_other_viral_days,key
0,2017-01-01,905,660.0,245.0,43,47,100,120,76,NaN,...,16.090096,2.096227,2.60465,5.144324,6.66377,4.170857,NaN,4.591475,5.386618,1


In [5]:
from datetime import timedelta

import yaml

from src.generators import sample_diurnal_arrival_seconds

with Path("data/assumptions.yaml").open(encoding="utf-8") as assumptions_file:
    arrival_settings = yaml.safe_load(assumptions_file)["appointment_arrivals"]

admission_columns = [column for column in df.columns if column.startswith("admissions_")]
source_groups = []

for _, source_row in df.iterrows():
    for column in admission_columns:
        category = column.removeprefix("admissions_")
        count = int(source_row[column]) if pd.notna(source_row[column]) else 0
        source_groups.append((source_row, category, count))

# ASSUMPTION: With no observed hourly data, all admission categories share this prototype diurnal profile.
arrival_seconds = sample_diurnal_arrival_seconds(
    [count for _, _, count in source_groups],
    seed=int(arrival_settings["seed"]),
    peak_hour=float(arrival_settings["peak_hour"]),
    amplitude=float(arrival_settings["amplitude"]),
)

appointment_records = []
arrival_offset = 0

for source_row, category, count in source_groups:
    group_times = arrival_seconds[arrival_offset : arrival_offset + count]
    for category_event_number, seconds_after_midnight in enumerate(group_times, start=1):
        appointment_records.append({
            "appointment_id": f"key-{int(source_row['key'])}-{category}-{category_event_number:04d}",
            "source_key": int(source_row["key"]),
            "date": source_row["date"],
            "category": category,
            "check_in_time": (
                pd.Timestamp(source_row["date"])
                + timedelta(seconds=int(seconds_after_midnight))
            ).time(),
        })
    arrival_offset += count

appointments = pd.DataFrame(appointment_records)
appointments = appointments.sort_values(["date", "check_in_time"]).reset_index(drop=True)

expected_count = int(df[admission_columns].fillna(0).to_numpy().sum())
assert len(appointments) == expected_count
assert appointments["appointment_id"].is_unique

print(f"Daily source rows: {len(df):,}")
print(f"Generated appointment records: {len(appointments):,}")
display(appointments["category"].value_counts().rename_axis("category").reset_index(name="appointment_count"))
display(appointments.head(10))

Daily source rows: 2,922
Generated appointment records: 372,197


,category,appointment_count
0,non_elective_via_ed,43800
1,non_elective_other,43742
2,newborn,43638
3,influenza,43577
4,delivery,43554
5,elective,43519
6,other_viral,43492
7,intensive,41674
8,covid_19,25201


,appointment_id,source_key,date,category,check_in_time
0,key-1-newborn-0015,1,2017-01-01,newborn,00:18:29
1,key-1-other_viral-0010,1,2017-01-01,other_viral,00:30:09
2,key-1-other_viral-0003,1,2017-01-01,other_viral,01:52:51
3,key-1-influenza-0006,1,2017-01-01,influenza,01:54:39
4,key-1-elective-0015,1,2017-01-01,elective,02:53:28
5,key-1-elective-0007,1,2017-01-01,elective,03:03:49
6,key-1-intensive-0012,1,2017-01-01,intensive,03:18:57
7,key-1-influenza-0009,1,2017-01-01,influenza,03:52:30
8,key-1-influenza-0002,1,2017-01-01,influenza,04:01:39
9,key-1-other_viral-0014,1,2017-01-01,other_viral,04:15:37


In [6]:
from src.generators import assign_patient_attributes

with Path("data/assumptions.yaml").open(encoding="utf-8") as assumptions_file:
    assumptions = yaml.safe_load(assumptions_file)
attribute_settings = assumptions["patient_attributes"]

# ASSUMPTION: severity, priority, and moveability are synthetic; see patient_attributes in assumptions.yaml.
counts_before = appointments["category"].value_counts()
appointments = assign_patient_attributes(
    appointments,
    attribute_settings,
    assumptions["nurse_staffing"]["category_to_unit"],
    seed=int(attribute_settings["seed"]),
)

assert appointments["category"].value_counts().equals(counts_before)
assert not ((appointments["category"] == "elective") & (appointments["severity"] == "critical")).any()
assert (appointments.loc[appointments["severity"] == "critical", "priority_rank"] == 1).all()
assert appointments.loc[appointments["is_moveable"], "move_window_days"].notna().all()
assert appointments.loc[~appointments["is_moveable"], "move_window_days"].isna().all()

ed_critical_share = (appointments.loc[appointments["category"] == "non_elective_via_ed", "severity"] == "critical").mean()
assert abs(ed_critical_share - 0.28) < 0.01, ed_critical_share
print(f"ED critical share: {ed_critical_share:.4f} (target 0.28)")

display(pd.crosstab(appointments["category"], appointments["severity"], normalize="index").round(3))
display(appointments.groupby("category")[["priority_rank", "is_moveable"]].agg(["min", "max"]))
display(pd.crosstab(appointments["category"], appointments["move_window_days"].astype("string").fillna("none")))

ED critical share: 0.2797 (target 0.28)


severity,low,medium,high,critical
category,,,,
covid_19,0.238,0.242,0.243,0.277
delivery,0.251,0.246,0.256,0.247
elective,0.333,0.335,0.332,0.000
influenza,0.240,0.240,0.237,0.284
intensive,0.000,0.000,0.502,0.498
newborn,0.247,0.254,0.250,0.250
non_elective_other,0.241,0.240,0.239,0.281
non_elective_via_ed,0.243,0.239,0.238,0.280
other_viral,0.238,0.240,0.241,0.281


priority_rank     is_moveable       
                              min max         min    max
category                                                
covid_19                        1   2       False   True
delivery                        1   1       False  False
elective                        4   4        True   True
influenza                       1   2       False   True
intensive                       1   1       False  False
newborn                         1   1       False  False
non_elective_other              1   3       False  False
non_elective_via_ed             1   2       False  False
other_viral                     1   2       False   True

move_window_days,0,14,30,7,none
category,,,,,
covid_19,18216,0,0,0,6985
delivery,0,0,0,0,43554
elective,0,14562,14487,14470,0
influenza,31222,0,0,0,12355
intensive,0,0,0,0,41674
newborn,0,0,0,0,43638
non_elective_other,0,0,0,0,43742
non_elective_via_ed,0,0,0,0,43800
other_viral,31286,0,0,0,12206


In [7]:
output_path = Path("data") / "synthetic_appointments.csv"
appointments.to_csv(output_path, index=False)
print(f"Saved {len(appointments):,} appointment records to {output_path}")

Saved 372,197 appointment records to data\synthetic_appointments.csv
